# 01 - Exploratory Data Analysis

Goal: understand the data before modeling. Three questions:

1. How imbalanced is the target?
2. Which columns have missing values?
3. Are there impossible or extreme values?

Every cleaning decision found here is written down with its reason.

## Setup

In [9]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

DATA_DIR = Path("../data/raw")

# Original Kaggle names are long and cryptic; meanings come from Data Dictionary.xls.
COLUMN_NAMES = {
    "SeriousDlqin2yrs": "defaulted",  # target: 90+ days past due within the next 2 years
    "RevolvingUtilizationOfUnsecuredLines": "credit_utilization",  # card/credit line balance / limits
    "age": "age",
    "NumberOfTime30-59DaysPastDueNotWorse": "late_30_59_days",  # times 30-59 days late, last 2 years
    "NumberOfTime60-89DaysPastDueNotWorse": "late_60_89_days",  # times 60-89 days late, last 2 years
    "NumberOfTimes90DaysLate": "late_90_plus_days",  # times 90+ days late
    "DebtRatio": "debt_ratio",  # monthly debt payments and living costs / monthly gross income
    "MonthlyIncome": "monthly_income",
    "NumberOfOpenCreditLinesAndLoans": "open_credit_lines_loans",
    "NumberRealEstateLoansOrLines": "real_estate_loans",
    "NumberOfDependents": "dependents",  # excluding the borrower
}
TARGET = "defaulted"

# The first CSV column is an unnamed row id, so use it as the index.
df = pd.read_csv(DATA_DIR / "cs-training.csv", index_col=0).rename(columns=COLUMN_NAMES)
df.shape

(150000, 11)

In [10]:
df.head()

,defaulted,credit_utilization,age,late_30_59_days,debt_ratio,monthly_income,open_credit_lines_loans,late_90_plus_days,real_estate_loans,late_60_89_days,dependents
1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 1 to 150000
Data columns (total 11 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   defaulted                150000 non-null  int64  
 1   credit_utilization       150000 non-null  float64
 2   age                      150000 non-null  int64  
 3   late_30_59_days          150000 non-null  int64  
 4   debt_ratio               150000 non-null  float64
 5   monthly_income           120269 non-null  float64
 6   open_credit_lines_loans  150000 non-null  int64  
 7   late_90_plus_days        150000 non-null  int64  
 8   real_estate_loans        150000 non-null  int64  
 9   late_60_89_days          150000 non-null  int64  
 10  dependents               146076 non-null  float64
dtypes: float64(4), int64(7)
memory usage: 12.6 MB


## 1. Target imbalance

What share of borrowers have `defaulted = 1`?


In [12]:
display(df[TARGET].value_counts())
display(df[TARGET].value_counts(normalize=True))

defaulted
0    139974
1     10026
Name: count, dtype: int64

defaulted
0    0.93316
1    0.06684
Name: proportion, dtype: float64

**Takeaway:** The data is imbalanced: 93.3% non-default and 6.7% default (10,026 of 150,000 borrowers).

- **Accuracy fails here.** The trivial baseline that predicts "nobody defaults" gets 93% accuracy, but recall = 0% (it never catches a default) and precision is undefined (0/0, it flags nobody).
- **We use AUC to compare models.** It does not depend on a threshold, and it measures how well the model ranks defaulters above non-defaulters. The trivial baseline gets only 0.5 (a coin flip), so AUC is not fooled the way accuracy is.
- **Consequences for later:** use stratified splits and stratified CV so every split keeps the 6.7% default rate, and set the decision threshold from the cost of each error (missed defaulter vs. rejected good customer), not the default 0.5.

## 2. Missing values

Which columns have missing values, and what share of rows?

*Hint:* `isna()` returns True/False; the mean of booleans is a proportion.

In [27]:
# missing data
missing_data = df.isna().mean()

display(missing_data[missing_data > 0].sort_values(ascending=False).round(4)*100)

monthly_income    19.82
dependents         2.62
dtype: float64

**Takeaway:** The missing data is concentrated on Monthly_income and dependents

Income has a lot of missing data, dropping them can cause a big distortion in prediction

In the otherside dependents is only at 2.6%



In [30]:
# Missing in both rows?

both_missing = (df['monthly_income'].isna() & df['dependents'].isna()).sum()

display(both_missing[both_missing>0])

array([3924])

**Takeaway:** All dependents missing also miss the montly_income ! There is no row where only dependents is missing

Thus, dependents are a subset of monthly_income

In [35]:
missing_income = df['monthly_income'].isna()

display(df.loc[missing_income,'debt_ratio'].describe())
display(df.loc[~missing_income, "debt_ratio"].describe())

count     29731.000000
mean       1673.396556
std        4248.372895
min           0.000000
25%         123.000000
50%        1159.000000
75%        2382.000000
max      329664.000000
Name: debt_ratio, dtype: float64

count    120269.000000
mean         26.598777
std         424.446457
min           0.000000
25%           0.143388
50%           0.296023
75%           0.482559
max       61106.500000
Name: debt_ratio, dtype: float64

In [36]:
df.groupby(missing_income)['debt_ratio'].describe()

,count,mean,std,min,25%,50%,75%,max
monthly_income,,,,,,,,
False,120269.0,26.598777,424.446457,0.0,0.143388,0.296023,0.482559,61106.5
True,29731.0,1673.396556,4248.372895,0.0,123.000000,1159.000000,2382.000000,329664.0


## 3. Impossible or extreme values

Look at the summary statistics. Check `age`, the three `late_*` columns, `debt_ratio` and `credit_utilization`.

*Hint:* `describe()` shows min/max; add `percentiles=[...]` to see the tails (e.g. 0.99).

In [16]:
# your code here

**Takeaway:** _list each suspicious value, why it is suspicious, and what you would do._